# StayIQ — Module 1: Booking Cancellation Prediction

This notebook contains the machine learning pipeline to build, train, and evaluate a binary classification model (XGBoost/LightGBM) predicting whether a guest will cancel their hotel reservation, based on the Kaggle *Hotel Booking Demand* schema.

### Pipeline Steps:
1. **Load Data**: Import raw Kaggle dataset.
2. **Data Cleaning**: Handle null values in `children`, `country`, and `agent` / `company` columns.
3. **Feature Engineering**: Derive features like total guests, duration of stay, and date-related attributes.
4. **Preprocessing**: Encode categorical fields (one-hot encoding) and scale numeric features.
5. **Train/Test Split**: Split data while maintaining class balance.
6. **Model Training**: Fit XGBoost and LightGBM classifiers.
7. **Evaluation**: Compute ROC-AUC, classification reports, and extract feature attributions.
8. **Export Model**: Serialize the trained model to `backend/app/models/` for backend inference.

In [ ]:
# 1. Imports and environment configurations
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix
import joblib

# Ensure models directory exists
os.makedirs('../backend/app/models', exist_ok=True)

In [ ]:
# 2. Load Dataset
# Note: Download 'hotel_bookings.csv' from Kaggle and place it in the 'data/' folder
data_path = "../data/hotel_bookings.csv"
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"Dataset loaded successfully. Shape: {df.shape}")
else:
    print("Dataset file not found. Generating simulated dataset for testing purposes...")
    # Generate dummy data matching Kaggle schema
    np.random.seed(42)
    size = 1000
    df = pd.DataFrame({
        'hotel': np.random.choice(['City Hotel', 'Resort Hotel'], size=size),
        'lead_time': np.random.randint(0, 365, size=size),
        'arrival_date_year': np.random.choice([2025, 2026], size=size),
        'arrival_date_month': np.random.choice(['July', 'August', 'September', 'October'], size=size),
        'arrival_date_week_number': np.random.randint(1, 53, size=size),
        'arrival_date_day_of_month': np.random.randint(1, 31, size=size),
        'stays_in_weekend_nights': np.random.randint(0, 4, size=size),
        'stays_in_week_nights': np.random.randint(0, 10, size=size),
        'adults': np.random.randint(1, 4, size=size),
        'children': np.random.randint(0, 3, size=size),
        'babies': np.random.randint(0, 2, size=size),
        'meal': np.random.choice(['BB', 'HB', 'FB', 'SC'], size=size),
        'country': np.random.choice(['PRT', 'GBR', 'ESP', 'FRA', 'DEU'], size=size),
        'market_segment': np.random.choice(['Online TA', 'Offline TA/TO', 'Direct', 'Corporate'], size=size),
        'distribution_channel': np.random.choice(['TA/TO', 'Direct', 'Corporate'], size=size),
        'is_repeated_guest': np.random.choice([0, 1], p=[0.95, 0.05], size=size),
        'previous_cancellations': np.random.choice([0, 1, 2], p=[0.9, 0.08, 0.02], size=size),
        'previous_bookings_not_canceled': np.random.randint(0, 5, size=size),
        'reserved_room_type': np.random.choice(['A', 'D', 'E', 'F'], size=size),
        'assigned_room_type': np.random.choice(['A', 'D', 'E', 'F'], size=size),
        'booking_changes': np.random.randint(0, 3, size=size),
        'deposit_type': np.random.choice(['No Deposit', 'Non Refund', 'Refundable'], p=[0.8, 0.19, 0.01], size=size),
        'agent': np.random.choice(['9', 'NULL', '240'], size=size),
        'company': np.random.choice(['NULL', '45'], size=size),
        'days_in_waiting_list': np.random.randint(0, 10, size=size),
        'customer_type': np.random.choice(['Transient', 'Transient-Party', 'Contract'], size=size),
        'adr': np.random.uniform(50.0, 250.0, size=size),
        'required_car_parking_spaces': np.random.choice([0, 1], p=[0.9, 0.1], size=size),
        'total_of_special_requests': np.random.randint(0, 4, size=size),
        'is_canceled': np.random.choice([0, 1], p=[0.63, 0.37], size=size)
    })
df.head()

In [ ]:
# 3. Data Preprocessing & Pipeline Construction

# Handle missing numerical values
df['children'] = df['children'].fillna(0)
df['country'] = df['country'].fillna('Undefined')

# Define feature categories
categorical_cols = ['hotel', 'arrival_date_month', 'meal', 'country', 
                    'market_segment', 'distribution_channel', 
                    'reserved_room_type', 'assigned_room_type', 
                    'deposit_type', 'customer_type']

numerical_cols = ['lead_time', 'arrival_date_year', 'arrival_date_week_number', 
                  'arrival_date_day_of_month', 'stays_in_weekend_nights', 
                  'stays_in_week_nights', 'adults', 'children', 'babies', 
                  'is_repeated_guest', 'previous_cancellations', 
                  'previous_bookings_not_canceled', 'booking_changes', 
                  'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 
                  'total_of_special_requests']

X = df[numerical_cols + categorical_cols]
y = df['is_canceled']

# Preprocessing transformers
numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [ ]:
# 4. Split and Train Classifier
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# For local running, try importing XGBoost. Fallback to Random Forest if missing.
try:
    from xgboost import XGBClassifier
    classifier = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='logloss')
    model_name = "XGBoost"
except ImportError:
    from sklearn.ensemble import RandomForestClassifier
    classifier = RandomForestClassifier(n_estimators=100, random_state=42)
    model_name = "Random Forest"

clf_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', classifier)])

print(f"Training model using {model_name}...")
clf_pipeline.fit(X_train, y_train)
print("Training completed.")

In [ ]:
# 5. Evaluate Model
y_pred = clf_pipeline.predict(X_test)
y_prob = clf_pipeline.predict_proba(X_test)[:, 1]

print(f"\n=== {model_name} Model Evaluation ===")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Active', 'Canceled'], yticklabels=['Active', 'Canceled'])
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# 6. Export Model for Backend Use
model_export_path = "../backend/app/models/cancellation_model.joblib"
joblib.dump(clf_pipeline, model_export_path)
print(f"Model pipeline exported successfully to: {model_export_path}")